In [ ]:
import msal

from dotmap import DotMap
import json
import os
from Api_Helpers import OneDriveConnector,DbConnector


connector = OneDriveConnector()
sql = DbConnector()


def iterate(root:str, item_id:str):
    print('')
    print(root, end='')
    connector.request_by_id(item_id)

    files = connector.files    
    for filex in files:
        file = DotMap(filex)
        fullname = os.path.join(root,file.name)
        if 'folder' in file:
            if file.name in ['node_modules','.git']: #or sql.exists_onedrive(fullname) :
                print('X',end='')
                continue
            print (f'#', end='')
            if not os.path.exists(fullname):
                os.makedirs(fullname)    
            iterate(fullname, file.id)
            #sql.add_id(fullname)
        elif 'file' in file:
            if sql.exists_onedrive(fullname) or 'AlbumArt' in file.name:
                print('-',end='')
                continue
            destination = os.path.join(root,file.name)
            if (not os.path.exists(destination) or (os.path.getsize(destination) == 0 and file.size > 0) )\
                and 'node_modules' not in destination:
                download = connector.request_by_url(file["@microsoft.graph.downloadUrl"])
                print()
                print(destination, end='')
                with open(destination, "wb") as f:
                    f.write(download.content)

            print (f'.', end='')
            sql.add_id(fullname)
        else: 
            print(f'{file.name} is locked')


response = connector.request_by_url(f'https://graph.microsoft.com/v1.0/me/drive/items/root/children')
files = response.json().get('value', [])    
for file in files:
   if file['name'] in ['Pictures','Videos']:
        print(file['name'], file['id'])
        path = f'/home/jordan/{file["name"]}'
        iterate(path, file['id'])



